# 01 - Scrape Kingshot Reviews

This notebook collects user reviews for **Kingshot** from the Google Play Store.

The default target is set to **10,000 reviews**. Scraping is performed incrementally and saved as a checkpoint so the process can be resumed if interrupted.

Main outputs:

```text
data/raw/kingshot_reviews_raw.csv
data/raw/kingshot_reviews_raw.xlsx
data/raw/kingshot_app_info.csv
```

Important notes:

- The notebook is stored in the `notebook/` directory.
- All outputs are written relative to the repository root using `BASE_DIR = Path("..").resolve()`.
- If scraping is interrupted, the latest checkpoint remains available.
- If a raw dataset already exists, the notebook loads it and appends newly collected reviews while removing duplicates.


## 1. Install Required Libraries

Run this cell only if the required packages are not installed. If `pip install -r requirements.txt` has already been executed, this cell can be skipped.


In [1]:
# Run if needed
# %pip install google-play-scraper pandas openpyxl

## 2. Import Libraries and Define Configuration

`TOTAL_REVIEW_TARGET = 10000` defines the maximum number of reviews to collect. For testing, reduce this value to `1000` or `3000`.


In [2]:
from pathlib import Path
import time
import pandas as pd
from google_play_scraper import app, Sort, reviews

APP_ID = "com.run.tower.defense"
APP_NAME = "kingshot"

TOTAL_REVIEW_TARGET = 14416
BATCH_SIZE = 200
DELAY_SECONDS = 1.5

LANG = "en"
COUNTRY = "us"

BASE_DIR = Path("..").resolve()
RAW_DIR = BASE_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE_CSV = RAW_DIR / f"{APP_NAME}_reviews_raw.csv"
RAW_FILE_XLSX = RAW_DIR / f"{APP_NAME}_reviews_raw.xlsx"

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR :", RAW_DIR)
print("Review target:", TOTAL_REVIEW_TARGET)

BASE_DIR: D:\Github\kingshot-review-sentiment-analysis
RAW_DIR : D:\Github\kingshot-review-sentiment-analysis\data\raw
Review target: 14416


## 3. Collect Application Metadata

The application metadata is collected to document the exact Google Play Store object used in this study.


In [3]:
try:
    app_info = app(APP_ID, lang=LANG, country=COUNTRY)
    app_summary = {
        "app_id": APP_ID,
        "title": app_info.get("title"),
        "developer": app_info.get("developer"),
        "score": app_info.get("score"),
        "ratings": app_info.get("ratings"),
        "reviews": app_info.get("reviews"),
        "installs": app_info.get("installs"),
        "genre": app_info.get("genre"),
        "url": app_info.get("url")
    }

    app_info_df = pd.DataFrame([app_summary])
    display(app_info_df)

    app_info_file = RAW_DIR / f"{APP_NAME}_app_info.csv"
    app_info_df.to_csv(app_info_file, index=False, encoding="utf-8-sig")

    print(f"Application metadata saved to: {app_info_file}")
except Exception as e:
    print("Failed to collect application metadata.")
    print("Error:", e)

,app_id,title,developer,score,ratings,reviews,installs,genre,url
0,com.run.tower.defense,Kingshot,Century Games PTE. LTD.,4.320533,1351967,14417,"10,000,000+",Strategy,https://play.google.com/store/apps/details?id=...


Application metadata saved to: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_app_info.csv


## 4. Load Existing Checkpoint

If a previous scraping checkpoint exists, it is loaded before collecting new reviews. Duplicate entries are removed using `reviewId` when available, or by combining `content` and `score`.


In [4]:
if RAW_FILE_CSV.exists():
    existing_df = pd.read_csv(RAW_FILE_CSV)
    print(f"Checkpoint found: {len(existing_df)} review")
else:
    existing_df = pd.DataFrame()
    print("No checkpoint found. Scraping starts from the beginning.")

existing_df.head()

Checkpoint found: 2200 review


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,4a782aca-25f9-4fcc-8909-085c56309635,Matt,https://play-lh.googleusercontent.com/a/ACg8oc...,once again the game is nothing like the ads!! ...,1,0,1.10.7,2026-06-09 06:12:39,NaN,NaN,1.10.7
1,00341945-4e30-4cb6-aecd-c450bc1c1ac4,Ryo Imaizumi,https://play-lh.googleusercontent.com/a/ACg8oc...,if you are familiar with rise of kingdoms or t...,1,0,NaN,2026-06-09 05:44:24,NaN,NaN,NaN
2,7d18e706-6f87-47df-b757-49d1a8f14926,Declan Chapman,https://play-lh.googleusercontent.com/a-/ALV-U...,This game has the most intrusive ads that forc...,1,0,NaN,2026-06-09 04:25:24,NaN,NaN,NaN
3,dd5afbde-4248-4867-ac93-37ec98bb3c0c,Joseph Culwell,https://play-lh.googleusercontent.com/a-/ALV-U...,this game is NOTHING like the ads. this compan...,1,0,1.10.7,2026-06-09 04:01:40,NaN,NaN,1.10.7
4,8adbbf7f-3b18-401e-9212-da8baee42733,Pool Gaming,https://play-lh.googleusercontent.com/a/ACg8oc...,is game ke bohat add ate Hein bekar gatia game...,1,0,1.10.7,2026-06-09 03:43:07,NaN,NaN,1.10.7


## 5. Incremental Review Scraping

The scraping process follows a checkpoint-based strategy:

- Retrieve reviews in batches using `BATCH_SIZE`.
- Save the dataset after each batch.
- Remove duplicate reviews at every checkpoint.
- Preserve the latest collected data if an error or interruption occurs.


In [5]:
all_reviews = []

if not existing_df.empty:
    all_reviews = existing_df.to_dict("records")

continuation_token = None
batch_number = 0

while len(all_reviews) < TOTAL_REVIEW_TARGET:
    batch_number += 1

    try:
        result, continuation_token = reviews(
            APP_ID,
            lang=LANG,
            country=COUNTRY,
            sort=Sort.NEWEST,
            count=min(BATCH_SIZE, TOTAL_REVIEW_TARGET - len(all_reviews)),
            continuation_token=continuation_token
        )

        if not result:
            print("No additional reviews were returned.")
            break

        all_reviews.extend(result)

        checkpoint_df = pd.DataFrame(all_reviews)

        if "reviewId" in checkpoint_df.columns:
            checkpoint_df = checkpoint_df.drop_duplicates(subset=["reviewId"])
        elif {"content", "score"}.issubset(checkpoint_df.columns):
            checkpoint_df = checkpoint_df.drop_duplicates(subset=["content", "score"])
        else:
            checkpoint_df = checkpoint_df.drop_duplicates()

        all_reviews = checkpoint_df.to_dict("records")

        checkpoint_df.to_csv(RAW_FILE_CSV, index=False, encoding="utf-8-sig")
        checkpoint_df.to_excel(RAW_FILE_XLSX, index=False)

        print(
            f"Batch {batch_number} selesai | "
            f"Total unik: {len(checkpoint_df)} | "
            f"CSV: {RAW_FILE_CSV}"
        )

        if continuation_token is None:
            print("Continuation token is no longer available.")
            break

        time.sleep(DELAY_SECONDS)

    except KeyboardInterrupt:
        print("Scraping was interrupted manually. The latest checkpoint has been saved.")
        break

    except Exception as e:
        print("Scraping stopped because of an error.")
        print("The latest data was saved if the previous batch completed successfully.")
        print("Error:", e)
        break

raw_df = pd.DataFrame(all_reviews)

if "reviewId" in raw_df.columns:
    raw_df = raw_df.drop_duplicates(subset=["reviewId"])
elif {"content", "score"}.issubset(raw_df.columns):
    raw_df = raw_df.drop_duplicates(subset=["content", "score"])
else:
    raw_df = raw_df.drop_duplicates()

raw_df.to_csv(RAW_FILE_CSV, index=False, encoding="utf-8-sig")
raw_df.to_excel(RAW_FILE_XLSX, index=False)

print("Scraping completed or stopped safely.")
print("Number of unique reviews:", len(raw_df))
print("Final CSV :", RAW_FILE_CSV)
print("Final Excel:", RAW_FILE_XLSX)

raw_df.head()

Batch 1 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 2 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 3 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 4 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 5 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 6 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 7 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 8 selesai | Total unik: 2200 | CSV: D:\Github\kingshot-review-sentiment-analysis\data\raw\kingshot_reviews_raw.csv
Batch 9 selesai | Total unik: 22

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,4a782aca-25f9-4fcc-8909-085c56309635,Matt,https://play-lh.googleusercontent.com/a/ACg8oc...,once again the game is nothing like the ads!! ...,1,0,1.10.7,2026-06-09 06:12:39,NaN,NaN,1.10.7
1,00341945-4e30-4cb6-aecd-c450bc1c1ac4,Ryo Imaizumi,https://play-lh.googleusercontent.com/a/ACg8oc...,if you are familiar with rise of kingdoms or t...,1,0,NaN,2026-06-09 05:44:24,NaN,NaN,NaN
2,7d18e706-6f87-47df-b757-49d1a8f14926,Declan Chapman,https://play-lh.googleusercontent.com/a-/ALV-U...,This game has the most intrusive ads that forc...,1,0,NaN,2026-06-09 04:25:24,NaN,NaN,NaN
3,dd5afbde-4248-4867-ac93-37ec98bb3c0c,Joseph Culwell,https://play-lh.googleusercontent.com/a-/ALV-U...,this game is NOTHING like the ads. this compan...,1,0,1.10.7,2026-06-09 04:01:40,NaN,NaN,1.10.7
4,8adbbf7f-3b18-401e-9212-da8baee42733,Pool Gaming,https://play-lh.googleusercontent.com/a/ACg8oc...,is game ke bohat add ate Hein bekar gatia game...,1,0,1.10.7,2026-06-09 03:43:07,NaN,NaN,1.10.7


## 6. Initial Data Inspection

This section reports the number of collected reviews and the initial rating distribution.


In [6]:
if not raw_df.empty:
    print("Number of reviews:", len(raw_df))

    if "score" in raw_df.columns:
        rating_counts = raw_df["score"].value_counts().sort_index().reset_index()
        rating_counts.columns = ["rating", "count"]
        display(rating_counts)

    display(raw_df.head())
else:
    print("The dataset is still empty.")

Number of reviews: 14600


,rating,count
0,1,6376
1,2,809
2,3,723
3,4,1148
4,5,5544


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,4a782aca-25f9-4fcc-8909-085c56309635,Matt,https://play-lh.googleusercontent.com/a/ACg8oc...,once again the game is nothing like the ads!! ...,1,0,1.10.7,2026-06-09 06:12:39,NaN,NaN,1.10.7
1,00341945-4e30-4cb6-aecd-c450bc1c1ac4,Ryo Imaizumi,https://play-lh.googleusercontent.com/a/ACg8oc...,if you are familiar with rise of kingdoms or t...,1,0,NaN,2026-06-09 05:44:24,NaN,NaN,NaN
2,7d18e706-6f87-47df-b757-49d1a8f14926,Declan Chapman,https://play-lh.googleusercontent.com/a-/ALV-U...,This game has the most intrusive ads that forc...,1,0,NaN,2026-06-09 04:25:24,NaN,NaN,NaN
3,dd5afbde-4248-4867-ac93-37ec98bb3c0c,Joseph Culwell,https://play-lh.googleusercontent.com/a-/ALV-U...,this game is NOTHING like the ads. this compan...,1,0,1.10.7,2026-06-09 04:01:40,NaN,NaN,1.10.7
4,8adbbf7f-3b18-401e-9212-da8baee42733,Pool Gaming,https://play-lh.googleusercontent.com/a/ACg8oc...,is game ke bohat add ate Hein bekar gatia game...,1,0,1.10.7,2026-06-09 03:43:07,NaN,NaN,1.10.7
